# Predição de Doença Cardíaca com Aprendizado de Máquina

**Dataset:** UCI Heart Disease – Cleveland

**Tarefa:** Classificação binária (0 = ausência, 1 = presença de doença cardíaca)

**Objetivo:** Comparar algoritmos de AM, otimizar hiperparâmetros, avaliar com métricas clínicas e interpretar o melhor modelo com SHAP.

**Reprodutibilidade:** seeds fixas, versões de bibliotecas registradas, notebook disponível.

In [ ]:
# =====================================================================
# 0. CONFIGURAÇÃO INICIAL
# =====================================================================
# Importa as bibliotecas necessárias para análise de dados, visualização,
# modelagem e interpretabilidade.
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES_DIR = ROOT_DIR / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------
# Reprodutibilidade
# ---------------------------------------------------------------
# Define uma seed global para que todos os processos aleatórios
# (divisão de dados, inicialização de modelos, etc.) sejam reproduzíveis.
SEED = 42
np.random.seed(SEED)

# ---------------------------------------------------------------
# Scikit-learn: pré-processamento, modelagem e métricas
# ---------------------------------------------------------------

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_predict, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve,
                             ConfusionMatrixDisplay)

# ---------------------------------------------------------------
# Algoritmos de classificação
# ---------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# ---------------------------------------------------------------
# Interpretabilidade
# ---------------------------------------------------------------
import shap

# ---------------------------------------------------------------
# Visualização: estilo dos gráficos
# ---------------------------------------------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# ---------------------------------------------------------------
# Versões das bibliotecas (para reprodutibilidade)
# ---------------------------------------------------------------
import sklearn, xgboost, joblib

print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost: {xgboost.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"shap: {shap.__version__}")
print(f"joblib: {joblib.__version__}")

## 1. Carregamento dos Dados

Baixa o dataset Cleveland do repositório UCI e separa o atributo alvo.

In [ ]:
# =====================================================================
# 1. CARREGAMENTO DOS DADOS
# =====================================================================
# O dataset é baixado diretamente da URL pública do UCI.

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Nomes das 14 colunas (13 atributos + 1 alvo)
colunas = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]

# Lê o CSV; '?' é convertido para NaN (valor ausente)
df = pd.read_csv(url, names=colunas, na_values='?')

# Visualiza as primeiras linhas para conferir o carregamento
df.head()

In [ ]:
# =====================================================================
# BINARIZAÇÃO DO ATRIBUTO ALVO
# =====================================================================
# O alvo original varia de 0 a 4 (gravidade da doença).
# Convertemos para binário: 0 = ausência, 1 = presença (1 a 4).

df['target'] = (df['num'] > 0).astype(int)

# Remove a coluna original 'num'
df = df.drop(columns=['num'])

# Verifica o formato final e a distribuição das classes
print("Formato do dataset:", df.shape)
print("\nDistribuição do alvo:")
print(df['target'].value_counts())
print("\nProporção das classes:")
print(df['target'].value_counts(normalize=True))

## 2. Análise Exploratória dos Dados (EDA)

- Tipos de atributos
- Valores faltantes
- Distribuição do alvo
- Estatísticas descritivas
- Histogramas e boxplots
- Correlações

In [ ]:
# =====================================================================
# 2. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)
# =====================================================================

# Informações gerais: tipos de dados e valores não nulos
print("Informações gerais do dataset:")
df.info()

# Estatísticas descritivas das variáveis numéricas
print("\nEstatísticas descritivas:")
df.describe().T

In [ ]:
# =====================================================================
# VERIFICAÇÃO DE VALORES FALTANTES
# =====================================================================
# Conta os valores ausentes por atributo e calcula o percentual.

missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100

# Tabela resumo apenas para atributos com valores faltantes
missing_df = pd.DataFrame({
    'Valores Faltantes': missing,
    'Percentual (%)': missing_pct
})
missing_df[missing_df['Valores Faltantes'] > 0]

In [ ]:
# =====================================================================
# VERIFICAÇÃO DE DUPLICATAS
# =====================================================================
# Conta os valores duplicados

print(f"\nLinhas duplicadas exatas: {df.duplicated().sum()}")

In [ ]:
# =====================================================================
# DISTRIBUIÇÃO DO ATRIBUTO ALVO
# =====================================================================
# Gráfico de barras mostrando o balanceamento entre as classes.

plt.figure(figsize=(7, 5))
sns.countplot(x='target', data=df)
plt.title('Distribuição do Alvo (0 = Ausência | 1 = Presença)')
plt.xlabel('Classe')
plt.ylabel('Número de Pacientes')
plt.xticks([0, 1], ['Sem doença', 'Com doença'])
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# HISTOGRAMAS DAS VARIÁVEIS NUMÉRICAS
# =====================================================================
# Mostra a distribuição de cada variável numérica.

num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

df[num_cols].hist(bins=20, figsize=(14, 10), edgecolor='black')
plt.suptitle('Histogramas das Variáveis Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# BOXPLOTS DAS VARIÁVEIS NUMÉRICAS POR CLASSE
# =====================================================================
# Compara a distribuição de cada variável entre pacientes com e sem doença.

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, col in enumerate(num_cols):
    sns.boxplot(x='target', y=col, data=df, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Classe')
    axes[i].set_ylabel('')
    axes[i].set_xticklabels(['Sem doença', 'Com doença'])

plt.suptitle('Distribuição das Variáveis Numéricas por Classe', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# MATRIZ DE CORRELAÇÃO
# =====================================================================
# Calcula as correlações entre todas as variáveis (Spearman).
# Serve para identificar relações lineares e possíveis redundâncias.

plt.figure(figsize=(8, 6))
corr = df[num_cols].corr(method="spearman")
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Matriz de Correlação de Spearman (Variáveis Numéricas)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# MATRIZ DE INFORMAÇÃO MÚTUA NORMALIZADA
# =====================================================================
# Calcula as "correlações" entre todas as variáveis categóricas
# Serve para identificar possíveis redundâncias.


from sklearn.metrics import normalized_mutual_info_score
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
ord_features = ['slope', 'ca'] 
nom_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'thal']  
df_cat_temp = df[nom_features + ord_features + ['target']].copy()
for col in df_cat_temp.columns:
    df_cat_temp[col] = df_cat_temp[col].fillna(df_cat_temp[col].mode()[0])

cols_cat = df_cat_temp.columns
matriz_nmi = pd.DataFrame(index=cols_cat, columns=cols_cat, dtype=float)

for c1 in cols_cat:
    for c2 in cols_cat:
        matriz_nmi.loc[c1, c2] = normalized_mutual_info_score(df_cat_temp[c1], df_cat_temp[c2])

plt.figure(figsize=(9, 7))
sns.heatmap(matriz_nmi, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1, square=True)
plt.title('Informação Mútua Normalizada (NMI) - Variáveis Categóricas e Alvo', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# DISTRIBUIÇÃO DAS VARIÁVEIS CATEGÓRICAS E ORDINAIS POR CLASSE
# =====================================================================
cat_plot_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(cat_plot_cols):
    sns.countplot(x=col, hue='target', data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} por Classe')
    axes[i].set_xlabel('')
    axes[i].legend(title='Doença', labels=['Não (0)', 'Sim (1)'])

plt.tight_layout()
plt.show()

## 3. Pré-processamento

- Separação entre X (atributos) e y (alvo)
- Divisão treino/teste estratificada (80/20)
- Pipeline de pré-processamento:
  - Numéricas: imputação pela mediana + padronização
  - Categóricas: imputação pela moda + one-hot encoding

In [ ]:
# =====================================================================
# 3. PRÉ-PROCESSAMENTO
# =====================================================================

# Separa atributos (X) e alvo (y)
X = df.drop(columns=['target'])
y = df['target']

# Divide em treino (80%) e teste (20%), mantendo a proporção das classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"\nProporção no treino:\n{y_train.value_counts(normalize=True)}")
print(f"\nProporção no teste:\n{y_test.value_counts(normalize=True)}")

In [ ]:
# =====================================================================
# DEFINIÇÃO DAS COLUNAS E DO PRÉ-PROCESSADOR (COM ORDINAIS)
# =====================================================================

num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
ord_features = ['slope', 'ca']  # Mantém a ordem clínica (1<2<3 e 0<1<2<3)
nom_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'thal']  # Sem ordem natural

# Numéricas: imputa pela mediana + padroniza (z-score)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Ordinais: imputa pela moda + codifica mantendo ordem + padroniza
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[[1.0, 2.0, 3.0], [0.0, 1.0, 2.0, 3.0]])),
    ('scaler', RobustScaler())
])

# Nominais: imputa pela moda + one-hot encoding
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('ord', ordinal_transformer, ord_features),
        ('nom', nominal_transformer, nom_features)
    ]
)

preprocessor

## 4. Spot-checking

Comparação inicial de 6 algoritmos com vieses indutivos diferentes,
usando validação cruzada estratificada de 5 folds.
Métricas: AUC-ROC, Recall, F1, Acurácia, Precisão.

In [ ]:
# =====================================================================
# 4. SPOT-CHECKING
# =====================================================================

# Dicionário com os 6 algoritmos a serem comparados
modelos = {
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED),
    'XGBoost': XGBClassifier(random_state=SEED, eval_metric='logloss'),
    'SVM (RBF)': SVC(probability=True, random_state=SEED),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

# Estratégia de validação cruzada: 5 folds estratificados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Métricas a serem calculadas em cada fold
scoring = {
    'roc_auc': 'roc_auc',
    'recall': 'recall',
    'f1': 'f1',
    'accuracy': 'accuracy',
    'precision': 'precision'
}

In [ ]:
# =====================================================================
# EXECUÇÃO DO SPOT-CHECKING
# =====================================================================
# Para cada algoritmo, treina e avalia via validação cruzada.
# A mesma seed é usada em todos os modelos para garantir comparabilidade.

resultados = []

for nome, modelo in modelos.items():
    # Pipeline que combina pré-processamento + classificador
    pipe = Pipeline(steps=[
        ('prep', preprocessor),
        ('clf', modelo)
    ])

    # Avaliação com validação cruzada
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=scoring, return_train_score=False
    )

    # Armazena as médias e desvio padrão dos resultados
    resultados.append({
        'Modelo': nome,
        'AUC-ROC': scores['test_roc_auc'].mean(),
        'AUC-ROC std': scores['test_roc_auc'].std(),
        'Recall': scores['test_recall'].mean(),
        'Recall std': scores['test_recall'].std(),
        'F1': scores['test_f1'].mean(),
        'F1 std': scores['test_f1'].std(),
        'Acurácia': scores['test_accuracy'].mean(),
        'Acurácia std': scores['test_accuracy'].std(),
        'Precisão': scores['test_precision'].mean(),
        'Precisão std': scores['test_precision'].std(),
    })

# Cria DataFrame ordenado por AUC-ROC (decrescente)
df_resultados = pd.DataFrame(resultados).sort_values('AUC-ROC', ascending=False)
df_resultados

In [ ]:
# =====================================================================
# GRÁFICO COMPARATIVO DO SPOT-CHECKING
# =====================================================================
# Mostra lado a lado as principais métricas para cada algoritmo.

df_resultados.set_index('Modelo')[['AUC-ROC', 'Recall', 'F1', 'Acurácia']].plot(
    kind='bar', figsize=(12, 6), edgecolor='black'
)
plt.title('Spot-checking: Comparação de Modelos', fontsize=13)
plt.ylabel('Valor da Métrica')
plt.xlabel('Algoritmo')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Otimização de Hiperparâmetros

Otimização dos 6 algoritmos.
Mantém-se o mesmo loop validação cruzada para escolher hiperparâmetros e seed para comparabilidade.

NestedCV separando a seleção de modelos (laço interno) e estimação de generalização (laço externo)

In [ ]:
# =====================================================================
#  NESTED CROSS-VALIDATION 
# =====================================================================
# Avalia o procedimento completo (pré-processamento + tuning interno)
# nos 6 algoritmos usando as mesmas partições externas.

espacos_busca = {
    'Regressão Logística': (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {'clf__C': [0.01, 0.1, 1, 10], 'clf__penalty': ['l2'], 'clf__class_weight': [None, 'balanced']}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=SEED),
        {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 5, 10], 'clf__min_samples_leaf': [1, 2, 4]}
    ),
    'XGBoost': (
        XGBClassifier(random_state=SEED, eval_metric='logloss'),
        {'clf__n_estimators': [100, 200], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.03, 0.1], 'clf__subsample': [0.8, 1.0]}
    ),
    'SVM (RBF)': (
        SVC(probability=True, random_state=SEED),
        {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.01, 0.1]}
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'clf__n_neighbors': [3, 5, 7, 11, 15], 'clf__weights': ['uniform', 'distance']}
    ),
    'Naive Bayes': (
        GaussianNB(),
        {'clf__var_smoothing': [1e-9, 1e-7, 1e-5]}
    )
}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)

nested_resultados = []
modelos_tunados_treino = {}

for nome, (estimador, grid) in espacos_busca.items():
    pipe = Pipeline(steps=[('prep', preprocessor), ('clf', estimador)])

    # Loop interno
    search = GridSearchCV(pipe, grid, cv=inner_cv, scoring='roc_auc', n_jobs=-1)
    
    # Loop externo do Nested CV
    scores = cross_validate(search, X_train, y_train, cv=outer_cv, scoring=scoring, n_jobs=-1)
    
    nested_resultados.append({
        'Modelo': nome,
        'AUC-ROC': scores['test_roc_auc'].mean(),
        'AUC-ROC std': scores['test_roc_auc'].std(),
        'Recall': scores['test_recall'].mean(),
        'Recall std': scores['test_recall'].std(),
        'F1': scores['test_f1'].mean(),
        'F1 std': scores['test_f1'].std(),
        'Acurácia': scores['test_accuracy'].mean(),
        'Acurácia std': scores['test_accuracy'].std(),
        'Precisão': scores['test_precision'].mean(),
        'Precisão std': scores['test_precision'].std(),
    })
    
    # Ajusta o GridSearchCV no treino completo para obter o estimador final de cada algoritmo
    search.fit(X_train, y_train)
    modelos_tunados_treino[nome] = search

df_nested = pd.DataFrame(nested_resultados).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
df_nested

In [ ]:
# =====================================================================
# SELEÇÃO DO MELHOR MODELO
# =====================================================================
# O melhor modelo é escolhido com base no AUC-ROC de validação cruzada.
# O conjunto de teste NÃO é usado nesta etapa.
melhor_nome = df_nested.loc[0, 'Modelo']
best_search = modelos_tunados_treino[melhor_nome]
best_model = best_search.best_estimator_

print(f"Melhor modelo (Nested CV): {melhor_nome}")
print(f"AUC-ROC (Nested CV): {df_nested.loc[0, 'AUC-ROC']:.4f} ± {df_nested.loc[0, 'AUC-ROC std']:.4f}")
print("Melhores hiperparâmetros no treino completo:", best_search.best_params_)

## 6. Avaliação Final

Avaliação do melhor modelo no conjunto de teste.

In [ ]:
# =====================================================================
# 6. AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# =====================================================================

# Predições: classe prevista e probabilidade da classe positiva
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Cálculo das métricas
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred):.4f}")

print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred,
                            target_names=['Sem doença', 'Com doença']))

In [ ]:
# =====================================================================
# MATRIZ DE CONFUSÃO
# =====================================================================
# Mostra a distribuição dos acertos e erros do modelo.

plt.figure(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Sem doença', 'Com doença'],
    cmap='Blues',
    colorbar=False
)
plt.title('Matriz de Confusão - Modelo Final', fontsize=13)
plt.tight_layout()
plt.show()



In [ ]:
# =====================================================================
# CURVA ROC DO MODELO FINAL
# =====================================================================

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_valor = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2,
         label=f'AUC = {auc_valor:.3f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório')
plt.xlabel('Taxa de Falso Positivo')
plt.ylabel('Taxa de Verdadeiro Positivo')
plt.title('Curva ROC - Modelo Final', fontsize=13)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# CURVA ROC COMPARATIVA ENTRE OS PRINCIPAIS MODELOS
# =====================================================================
plt.figure(figsize=(9, 7))
for nome, search_obj in modelos_tunados_treino.items():
    modelo = search_obj.best_estimator_
    y_proba_i = modelo.predict_proba(X_test)[:, 1]
    fpr_i, tpr_i, _ = roc_curve(y_test, y_proba_i)
    auc_i = roc_auc_score(y_test, y_proba_i)
    plt.plot(fpr_i, tpr_i, linewidth=2, label=f'{nome} (AUC = {auc_i:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório')
plt.xlabel('Taxa de Falso Positivo')
plt.ylabel('Taxa de Verdadeiro Positivo')
plt.title('Curva ROC no Teste - Comparação de Modelos Otimizados', fontsize=13)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Interpretabilidade

Aplicação do SHAP para identificar os atributos mais relevantes.
O explainer é escolhido de acordo com o tipo do modelo.

In [ ]:
# =====================================================================
# 7. INTERPRETABILIDADE - PREPARAÇÃO
# =====================================================================

# Extrai o pré-processador e o classificador do pipeline
prep = best_model.named_steps['prep']
clf = best_model.named_steps['clf']

# Transforma os conjuntos de treino e teste usando o pré-processador
X_train_prep = prep.transform(X_train)
X_test_prep = prep.transform(X_test)

# Obtém os nomes das features após o one-hot encoding
feature_names = prep.get_feature_names_out()
print(f"Número de features após pré-processamento: {len(feature_names)}")

In [ ]:
# =====================================================================
# 7.1 SHAP - VERSÃO ADAPTATIVA
# =====================================================================
# Escolhe o explainer correto de acordo com o tipo de modelo.

tipo_modelo = type(clf).__name__
print(f"Tipo do classificador: {tipo_modelo}")

if tipo_modelo in ['RandomForestClassifier', 'XGBClassifier',
                   'GradientBoostingClassifier', 'DecisionTreeClassifier',
                   'ExtraTreesClassifier']:
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test_prep)
elif tipo_modelo in ['LogisticRegression', 'LinearSVC', 'Ridge', 'Lasso']:
    explainer = shap.LinearExplainer(clf, X_train_prep)
    shap_values = explainer.shap_values(X_test_prep)
else:
    explainer = shap.KernelExplainer(clf.predict_proba, X_train_prep[:100])
    shap_values = explainer.shap_values(X_test_prep[:100])

print(f"Tipo: {type(shap_values)}")
print(f"Shape: {np.shape(shap_values)}")

# Normaliza para array 2D (amostras x features) da classe positiva
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_pos = shap_values[:, :, 1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 2:
    shap_values_pos = shap_values
else:
    raise ValueError(f"Formato não reconhecido: {np.shape(shap_values)}")

print(f"Shape shap_values_pos: {shap_values_pos.shape}")
print(f"Shape X_test_prep: {X_test_prep.shape}")

In [ ]:
# =====================================================================
# 7.2 SHAP - SUMMARY PLOT (BEESWARM)
# =====================================================================
# Mostra o impacto de cada atributo nas predições.

shap.summary_plot(shap_values_pos, X_test_prep, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================================
# 7.3 SHAP - IMPORTÂNCIA GLOBAL
# =====================================================================
# Mostra a importância média absoluta de cada atributo.

shap.summary_plot(shap_values_pos, X_test_prep,
                  feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig( FIGURES_DIR / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================================
# 7.4 ATRIBUTO MAIS IMPORTANTE SEGUNDO SHAP
# =====================================================================

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
top_feature_idx = np.argmax(mean_abs_shap)
top_feature_name = feature_names[top_feature_idx]
print(f"Atributo mais importante: {top_feature_name}")

In [ ]:
# =====================================================================
# 7.5 SHAP - DEPENDENCE PLOT
# =====================================================================
# Mostra como o valor do atributo mais importante afeta a predição.

shap.dependence_plot(top_feature_idx, shap_values_pos, X_test_prep,
                     feature_names=feature_names)
plt.tight_layout()
plt.show()

## 8. Importância por Permutação

Método complementar ao SHAP para validar a relevância dos atributos.

In [ ]:
# =====================================================================
# 8. IMPORTÂNCIA POR PERMUTAÇÃO
# =====================================================================
# Mede quanto o desempenho cai quando cada atributo é embaralhado.

from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=30, random_state=SEED, scoring='roc_auc'
)

importancias = pd.DataFrame({
    'Atributo': X_test.columns,
    'Importância Média': result.importances_mean,
    'Desvio Padrão': result.importances_std
}).sort_values('Importância Média', ascending=False).reset_index(drop=True)

print(importancias.head(10))

In [ ]:
# =====================================================================
# GRÁFICO DA IMPORTÂNCIA POR PERMUTAÇÃO
# =====================================================================

top10 = importancias.head(10).iloc[::-1]  # Inverte para ordenar no gráfico

plt.figure(figsize=(10, 7))
plt.barh(top10['Atributo'], top10['Importância Média'],
         xerr=top10['Desvio Padrão'], color='steelblue', edgecolor='black')
plt.xlabel('Importância por Permutação (Δ AUC-ROC)')
plt.title('Top 10 Atributos - Importância por Permutação', fontsize=13)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Análise de Threshold

Avaliação de como recall, precisão e F1 variam conforme o ponto de corte.

In [ ]:
# =====================================================================
# 9. ANÁLISE DE THRESHOLD 
# =====================================================================
# Atualizado para ser fora do conjunto de testes para evitar data leakage.
y_train_proba_oof = cross_val_predict(
    best_model, X_train, y_train, cv=outer_cv, method='predict_proba', n_jobs=-1
)[:, 1]

thresholds = np.arange(0.1, 1.0, 0.05)
resultados_thr = []

for thr in thresholds:
    y_pred_thr = (y_train_proba_oof >= thr).astype(int)
    resultados_thr.append({
        'Threshold': thr,
        'Recall': recall_score(y_train, y_pred_thr),
        'Precisão': precision_score(y_train, y_pred_thr),
        'F1': f1_score(y_train, y_pred_thr)
    })

df_thr = pd.DataFrame(resultados_thr)
print(df_thr)

In [ ]:
# =====================================================================
# GRÁFICO DE MÉTRICAS VS THRESHOLD
# =====================================================================

plt.figure(figsize=(10, 6))
plt.plot(df_thr['Threshold'], df_thr['Recall'], label='Recall',
         color='steelblue', linewidth=2)
plt.plot(df_thr['Threshold'], df_thr['Precisão'], label='Precisão',
         color='darkorange', linewidth=2)
plt.plot(df_thr['Threshold'], df_thr['F1'], label='F1',
         color='green', linewidth=2)
plt.axvline(0.5, color='gray', linestyle='--',
            label='Threshold padrão (0.5)')
plt.xlabel('Threshold de Decisão')
plt.ylabel('Valor da Métrica')
plt.title('Métricas vs Threshold de Decisão', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Reprodutibilidade

- Seeds fixas em todas as etapas
- Versões de bibliotecas registradas
- Modelo final salvo
- Resultados exportados em CSV

In [ ]:
# =====================================================================
# 10. SALVAMENTO DE MODELO E RESULTADOS
# =====================================================================

import json
import joblib
import pandas as pd

# Define os caminhos a partir da raiz do projeto (um nível acima de notebooks/)
# ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MODELS_DIR = ROOT_DIR / "models"
PARAMS_DIR = MODELS_DIR / "params"
METRICS_DIR = ROOT_DIR / "reports" / "metrics"
# FIGURES_DIR = ROOT_DIR / "reports" / "figures"

# Cria os diretórios caso ainda não existam
for path in [MODELS_DIR, PARAMS_DIR, METRICS_DIR, FIGURES_DIR]:
  path.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 1. Métricas e Tabelas (reports/metrics/)
# ---------------------------------------------------------------------

metricas_finais = {
    "Modelo Escolhido": melhor_nome,
    "AUC-ROC Nested CV (Média)": df_nested.loc[0, 'AUC-ROC'],
    "AUC-ROC Nested CV (Std)": df_nested.loc[0, 'AUC-ROC std'],
    "AUC-ROC Teste": roc_auc_score(y_test, y_proba),
    "Recall Teste": recall_score(y_test, y_pred),
    "F1 Teste": f1_score(y_test, y_pred),
    "Acurácia Teste": accuracy_score(y_test, y_pred),
    "Precisão Teste": precision_score(y_test, y_pred),
}

pd.DataFrame([metricas_finais]).to_csv(METRICS_DIR / "metricas_finais.csv", index=False)
df_resultados.to_csv(METRICS_DIR / "resultados_spot_checking.csv", index=False)
df_nested.to_csv(METRICS_DIR / "resultados_nested_cv.csv", index=False)
importancias.to_csv(METRICS_DIR / "permutation_importance.csv", index=False)


# ---------------------------------------------------------------------
# 2. Modelo Treinado (models/)
# ---------------------------------------------------------------------
joblib.dump(best_model, MODELS_DIR / "melhor_modelo_heart_disease.pkl")

# ---------------------------------------------------------------------
# 3. Hiperparâmetros Ótimos (models/params/)
# ---------------------------------------------------------------------
melhores_params_todos = {nome: obj.best_params_ for nome, obj in modelos_tunados_treino.items()}
with open(PARAMS_DIR / "best_params_todos_modelos.json", "w", encoding="utf-8") as f:
    json.dump(melhores_params_todos, f, indent=2, default=str)

print(f"Artefatos salvos com sucesso em:\n - {MODELS_DIR}\n - {METRICS_DIR}")

In [ ]:
# =====================================================================
# RESUMO FINAL
# =====================================================================

print("=" * 60)
print("RESUMO FINAL DO TRABALHO")
print("=" * 60)
print(f"\nMelhor modelo: {melhor_nome}")
print(f"\nMétricas no conjunto de teste:")
for k, v in metricas_finais.items():
    if k != 'Modelo Escolhido':
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

print("\nArquivos salvos:")
for arq in [
    'shap_summary.png',
    'shap_bar.png',
    'permutation_importance.png',
    'metricas_finais.csv',
    'resultados_spot_checking.csv',
    'resultados_nested_cv.csv',
    'permutation_importance.csv',
    'melhor_modelo_heart_disease.pkl',
    'best_params_todos_modelos.json'
]:
    print(f"  - {arq}")
